# Fundas_System -- Glaucoma Specialist Training (Phase 3, plans/prep.txt)

FUNDAS_SYSTEM -- TRAIN PHASE 3 (GLAUCOMA CHUYEN BIET) TREN GOOGLE COLAB
====================================================
Boi canh: plans/plan.txt Phase 3 + plans/prep.txt (chuan bi kha thi, 2026-08-30).
Model nay la 1 bo phan loai NHI PHAN rieng (glaucoma / khong glaucoma), tach
biet voi multidisease baseline (Phase 2) -- xem plans/plan_multidisease.txt
de biet ly do 2 tinh nang nay khong gop chung.

Dataset: ghep 3 nguon cong khai qua Kaggle API (da xac nhan kha dung +
license qua `kaggle datasets metadata`, xem plans/prep.txt muc 1):
  - ACRIMA      (orvile/acrima-glaucoma-assessment-using-fundus-images)   705 anh (396 glaucoma / 309 normal), CC BY 4.0
  - ORIGA       (ferencjuhsz/origa-retinal-fundus-image-dataset)          650 anh (168 glaucoma / 482 normal), nhan "MIT" tren Kaggle KHONG dang tin cay (xem prep.txt muc 1 -- co the la nhan tu-gan cua nguoi upload)
  - RIM-ONE DL  (orvile/rim-one-retinal-dataset-for-assessing-glaucoma)   485 anh (172 glaucoma / 313 normal), dieu khoan GOC la non-commercial/research-only (Kaggle ghi CC BY 4.0 nhung mo ta chinh dataset noi khac -- xem prep.txt)
  Tong: ~1840 anh, ty le 40% glaucoma / 60% normal -- can bang hon nhieu
  so class imbalance da gap o ODIR-5K (Phase 2).

LUU Y LICENSE: ORIGA va RIM-ONE thuc chat la non-commercial/research-only
du Kaggle gan nhan khac -- NGUOI DUNG DA XAC NHAN (2026-08-30) day la do an
CA NHAN, KHONG kinh doanh, nen chap nhan dung ca 3 bo. Neu du an nay sau
nay thuong mai hoa, PHAI doc lai plans/prep.txt muc 1 truoc.

CAU TRUC FILE THAT (da xac nhan qua `kaggle datasets files`/API list,
KHONG doan mo):
  - ACRIMA:  Database/Images/Im<3-so>_ACRIMA.jpg -- nhan nam TRONG TEN FILE:
             co "_g_" = glaucoma, khong co = normal (vd Im686_g_ACRIMA.jpg).
  - ORIGA:   ORIGA_train.csv + ORIGA_test.csv, cot `ImageName,glaucoma`
             (glaucoma la 0/1), ImageName da co san duong dan tuong doi
             (vd "ORIGA/train/125.jpg").
  - RIM-ONE: RIM-ONE_DL_images/partitioned_randomly/{training_set,test_set}/
             {glaucoma,normal}/<file>.png -- nhan nam trong TEN THU MUC.

Kien truc code (crop/normalize) PHAI khop inference/preprocessing.py de
sau nay viet inference/glaucoma_engine.py doc checkpoint nay khong bi lech
phan phoi du lieu so voi luc train -- xem lai cach da lam voi multidisease
(plans/ggcolab.txt).

Output cua notebook nay (2 file tai ve rieng, tu dong khi chay het cell):
  1. model_best.pth -- checkpoint tot nhat (chon theo val F1 lop glaucoma).
  2. glaucoma_test_samples.zip -- 10 anh ngau nhien lay tu tap validation
     (chua qua model nhin thay khi train) kem file labels that (JSON) de
     kiem thu cuc bo sau khi wire xong inference/glaucoma_engine.py.

CHUAN BI TRUOC KHI CHAY:
  1. Tai kaggle.json (dang dung o C:\Users\ADMIN\.kaggle\kaggle.json local)
     len Google Drive, vi du: /content/drive/MyDrive/kaggle.json
  2. Colab: Runtime > Change runtime type > GPU (T4 la du, dataset nho hon
     ODIR-5K nhieu nen se nhanh hon Phase 2).
  3. Chay lan luot tung cell TU TREN XUONG.

## CELL 1: Mount Drive + cau hinh Kaggle API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials ready.")

## CELL 2: Cai dependency + tai 3 dataset glaucoma

In [ ]:
!pip install -q kaggle timm

import kaggle
kaggle.api.authenticate()

DATASETS = {
    'acrima': 'orvile/acrima-glaucoma-assessment-using-fundus-images',
    'origa': 'ferencjuhsz/origa-retinal-fundus-image-dataset',
    'rimone': 'orvile/rim-one-retinal-dataset-for-assessing-glaucoma',
}

for name, ref in DATASETS.items():
    print(f'Downloading {name} ({ref})...')
    kaggle.api.dataset_download_files(ref, path=f'/content/glaucoma_raw/{name}', unzip=True, quiet=False)

print('All datasets downloaded.')

## CELL 3: Imports + tien xu ly (khop inference/preprocessing.py)

Ham crop/normalize duoi day la ban sao chinh xac cua inference/preprocessing.py
(load_and_crop, preprocess_for_backbone) de checkpoint sau nay khop dung
phan phoi du lieu voi inference/ local -- KHONG duoc tu tien doi logic o day
ma khong doi ca file local.

In [ ]:
import json
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

RAW_ROOT = Path('/content/glaucoma_raw')
BACKBONE = 'seresnext50_32x4d'   # dung lai backbone da co trong DR/multidisease
IMAGE_SIZE = 224
DROPOUT = 0.3
VAL_FRACTION = 0.15
SEED = 0
EPOCHS = 20
BATCH_SIZE = 32
LR = 3e-4
N_TEST_SAMPLES = 10               # so anh xuat ra de test local (Cell 8)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def crop_dark_borders(img: np.ndarray, tol: int = 7) -> np.ndarray:
    """Y het inference/preprocessing.py::_crop_dark_borders."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if mask.sum() == 0:
        return img
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    cropped = img[rows[0]: rows[-1] + 1, cols[0]: cols[-1] + 1]
    return cropped if cropped.size else img


def load_and_crop(image_path) -> np.ndarray:
    """Y het inference/preprocessing.py::load_and_crop."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f'Could not read image: {image_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return crop_dark_borders(img)


_MEAN, _STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet fallback
try:
    cfg = timm.get_pretrained_cfg(BACKBONE)
    if cfg is not None and cfg.mean and cfg.std:
        _MEAN, _STD = cfg.mean, cfg.std
except Exception:
    pass

_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_MEAN, std=_STD),
])


def preprocess(cropped_rgb: np.ndarray) -> torch.Tensor:
    """Y het inference/preprocessing.py::preprocess_for_backbone."""
    return _TRANSFORM(Image.fromarray(cropped_rgb))


# Chan doan nhanh: in kich thuoc anh goc cua vai anh moi nguon, vi ACRIMA co
# the da duoc crop san quanh dia thi giac (khac fundus tron day du) -- xem
# plans/prep.txt muc 2.1. Neu ty le W/H cua ACRIMA khac han 2 nguon con lai,
# ghi nhan lai (khong chan qua trinh chay, chi de biet truoc khi tin ket qua).
def _peek_sizes(pattern, label, n=5):
    paths = list(RAW_ROOT.glob(pattern))[:n]
    sizes = [Image.open(p).size for p in paths]
    print(f'  {label}: {sizes}')

print('Kich thuoc anh mau (W, H) moi nguon (kiem tra nhanh quy uoc crop):')
_peek_sizes('acrima/**/Im*_ACRIMA.jpg', 'ACRIMA')
_peek_sizes('origa/ORIGA/train/*.jpg', 'ORIGA')
_peek_sizes('rimone/**/glaucoma/*.png', 'RIM-ONE')

## CELL 4: Gop nhan tu 3 nguon thanh 1 manifest chung

Moi nguon co quy uoc nhan khac nhau (xem docstring dau file) -- gop thanh
list (filepath, label:int, source:str) dong nhat, label 1=glaucoma,
0=normal.

In [ ]:
def load_acrima():
    rows = []
    for p in sorted((RAW_ROOT / 'acrima').rglob('*_ACRIMA.jpg')):
        label = 1 if '_g_' in p.name.lower() else 0
        rows.append((p, label, 'acrima'))
    return rows


def load_origa():
    rows = []
    origa_dir = RAW_ROOT / 'origa'
    for csv_name in ('ORIGA_train.csv', 'ORIGA_test.csv'):
        df = pd.read_csv(origa_dir / csv_name)
        for _, row in df.iterrows():
            # ImageName da co san duong dan tuong doi vd "ORIGA/train/125.jpg"
            path = origa_dir / row['ImageName']
            if not path.exists():
                continue
            rows.append((path, int(row['glaucoma']), 'origa'))
    return rows


def load_rimone():
    rows = []
    base = RAW_ROOT / 'rimone' / 'RIM-ONE_DL_images' / 'partitioned_randomly'
    for split in ('training_set', 'test_set'):
        for label_name, label in (('glaucoma', 1), ('normal', 0)):
            for p in sorted((base / split / label_name).glob('*.png')):
                rows.append((p, label, 'rimone'))
    return rows


all_rows = load_acrima() + load_origa() + load_rimone()
print(f'Total images loaded: {len(all_rows)}')

import collections
per_source = collections.Counter(r[2] for r in all_rows)
per_label = collections.Counter(r[1] for r in all_rows)
per_source_label = collections.Counter((r[2], r[1]) for r in all_rows)
print('Per source:', dict(per_source))
print('Per label (1=glaucoma):', dict(per_label))
print('Per source x label:', dict(per_source_label))

assert len(all_rows) > 1500, 'Expected ~1840 images total -- check dataset paths above if this fails.'

## CELL 5: Split train/val (stratify theo label VA nguon) + Dataset/DataLoader

Khong co patient/eye ID trong ACRIMA/ORIGA/RIM-ONE (khac ODIR-5K Phase 2,
xem plans/prep.txt muc 2.2) nen split o day la THEO TUNG ANH, khong theo
benh nhan. Stratify theo (label, source) thay vi chi theo label, de tranh
tinh huong 1 nguon (vd ORIGA - Singapore) bi don het vao 1 phia train/val
va gay lech domain khi danh gia (xem prep.txt muc 2.3).

In [ ]:
def stratified_split(rows, val_fraction, seed):
    rng = random.Random(seed)
    groups = collections.defaultdict(list)
    for row in rows:
        groups[(row[1], row[2])].append(row)

    train_rows, val_rows = [], []
    for key, group in groups.items():
        group = group[:]
        rng.shuffle(group)
        n_val = max(1, round(len(group) * val_fraction))
        val_rows.extend(group[:n_val])
        train_rows.extend(group[n_val:])
    return train_rows, val_rows


train_rows, val_rows = stratified_split(all_rows, VAL_FRACTION, SEED)
print(f'Train: {len(train_rows)}  Val: {len(val_rows)}')
print('Val per source x label:', dict(collections.Counter((r[2], r[1]) for r in val_rows)))


class GlaucomaDataset(Dataset):
    def __init__(self, rows, augment: bool):
        self.rows = rows
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        path, label, _source = self.rows[idx]
        cropped = load_and_crop(path)
        if self.augment and np.random.rand() < 0.5:
            cropped = np.ascontiguousarray(cropped[:, ::-1])
        return preprocess(cropped), torch.tensor(label, dtype=torch.float32)


train_labels = np.array([r[1] for r in train_rows])
n_pos, n_neg = train_labels.sum(), len(train_labels) - train_labels.sum()
pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32)
print(f'Train pos(glaucoma)={int(n_pos)} neg(normal)={int(n_neg)} pos_weight={pos_weight.item():.3f}')

train_loader = DataLoader(GlaucomaDataset(train_rows, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(GlaucomaDataset(val_rows, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## CELL 6: Model -- phan loai nhi phan (1 logit, sigmoid = P(glaucoma))

Cung mau voi inference/multidisease_model.py (1 backbone timm + head Linear
don, KHONG ensemble 4x5 fold nhu DR -- tap qua nho de train 20 model rieng,
xem plans/prep.txt muc 3). Khac multidisease o cho day la BINARY (1 logit +
BCEWithLogitsLoss) thay vi 8-lop CrossEntropy.

In [ ]:
class GlaucomaModel(nn.Module):
    def __init__(self, backbone_name, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)  # raw logit


model = GlaucomaModel(BACKBONE, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

## CELL 7: Vong lap train -- chon model tot nhat theo val F1 (lop glaucoma)

Dung F1 cua lop glaucoma (khong phai accuracy tho) de chon best checkpoint,
vi day la boi toan sang loc benh -- bo sot glaucoma (false negative) nghiem
trong hon nham normal thanh glaucoma. Sau vong lap, in them accuracy RIENG
theo tung nguon dataset de phat hien neu model chi hoc tot 1 nguon (domain
shift, xem plans/prep.txt muc 2.3).

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        probs = torch.sigmoid(model(x)).cpu()
        all_probs.append(probs)
        all_labels.append(y)
    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = (probs >= 0.5).astype(int)

    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (preds == labels).mean()
    return {'accuracy': float(accuracy), 'precision': precision, 'recall': recall, 'f1': f1}


def evaluate_per_source(model, rows):
    by_source = collections.defaultdict(list)
    for row in rows:
        by_source[row[2]].append(row)
    results = {}
    for source, source_rows in by_source.items():
        loader = DataLoader(GlaucomaDataset(source_rows, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        results[source] = evaluate(model, loader)
    return results


best_f1 = -1.0
best_state = None
best_metrics = {}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_rows)
    metrics = evaluate(model, val_loader)
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_acc={metrics['accuracy']:.3f}  val_precision={metrics['precision']:.3f}  "
          f"val_recall={metrics['recall']:.3f}  val_f1={metrics['f1']:.3f}")

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_metrics = {'val_accuracy': metrics['accuracy'], 'val_precision': metrics['precision'],
                         'val_recall': metrics['recall'], 'val_f1': metrics['f1'], 'epoch': epoch}

print(f"\nBest: val_f1={best_f1:.3f}  (epoch {best_metrics.get('epoch')})")

# Danh gia rieng tung nguon bang model TOT NHAT (khong phai model cuoi cung)
model.load_state_dict(best_state)
print('\nPer-source breakdown (best checkpoint):')
for source, m in evaluate_per_source(model, val_rows).items():
    print(f"  {source:8s}  acc={m['accuracy']:.3f}  precision={m['precision']:.3f}  recall={m['recall']:.3f}  f1={m['f1']:.3f}")

## CELL 8: Luu checkpoint + TU DONG tai ve may

Chay xong cell nay se tu dong: (1) luu vao Google Drive, (2) trigger tai ve
trinh duyet (`files.download`) -- khong can buoc thu cong nao them.

In [ ]:
CHECKPOINT_NAME = 'model_best.pth'
torch.save(
    {
        'model_state_dict': best_state,
        'labels': ['normal', 'glaucoma'],   # index 1 = positive class cua sigmoid logit
        'backbone': BACKBONE,
        'dropout': DROPOUT,
        'sources': sorted(per_source.keys()),
        **best_metrics,
    },
    CHECKPOINT_NAME,
)

shutil.copy(CHECKPOINT_NAME, '/content/drive/MyDrive/' + CHECKPOINT_NAME)
print('Saved to Google Drive:', '/content/drive/MyDrive/' + CHECKPOINT_NAME)

from google.colab import files
files.download(CHECKPOINT_NAME)   # tu dong tai ve may

print(json.dumps({
    'task': 'Glaucoma Specialist Screening (Phase 3, plans/prep.txt)',
    'dataset_sources': {
        'acrima': 'https://www.kaggle.com/datasets/orvile/acrima-glaucoma-assessment-using-fundus-images',
        'origa': 'https://www.kaggle.com/datasets/ferencjuhsz/origa-retinal-fundus-image-dataset',
        'rimone': 'https://www.kaggle.com/datasets/orvile/rim-one-retinal-dataset-for-assessing-glaucoma',
    },
    'backbone': BACKBONE,
    'image_size': IMAGE_SIZE,
    'dropout': DROPOUT,
    'labels': ['normal', 'glaucoma'],
    'checkpoint': CHECKPOINT_NAME,
    **best_metrics,
}, indent=4))

## CELL 9: Xuat 10 anh ngau nhien, TY LE 5 glaucoma / 5 normal (tu tap validation) de test local

Lay tu VAL SET (model chua bao gio thay khi train) chu khong phai train
set, de con y nghia lam bo test cuc bo. Sample RIENG 5 tu nhom glaucoma va
5 tu nhom normal (khong random tho tren ca pool, vi pool lech 40/60 nen
random thuong se KHONG ra dung 5-5) roi tron thu tu lai. Kem 1 file labels
that (JSON) vi khong co nhan that thi khong kiem chung duoc du doan cua
inference engine sau nay. Copy ANH GOC (chua qua crop/resize) de test dung
duong di thuc te (upload -> load_and_crop -> ...), khong phai tensor da
tien xu ly.

In [ ]:
SAMPLE_DIR = Path('/content/glaucoma_test_samples')
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
SAMPLE_DIR.mkdir(parents=True)

rng = random.Random(SEED + 1)
glaucoma_rows = [r for r in val_rows if r[1] == 1]
normal_rows = [r for r in val_rows if r[1] == 0]

n_each = N_TEST_SAMPLES // 2  # 5 + 5
sample_rows = (
    rng.sample(glaucoma_rows, min(n_each, len(glaucoma_rows)))
    + rng.sample(normal_rows, min(n_each, len(normal_rows)))
)
rng.shuffle(sample_rows)  # tron thu tu, khong de 5 anh dau toan glaucoma

label_names = {0: 'normal', 1: 'glaucoma'}
manifest = []
for i, (path, label, source) in enumerate(sample_rows, start=1):
    dest_name = f'{i:02d}_{source}_{label_names[label]}{path.suffix}'
    shutil.copy(path, SAMPLE_DIR / dest_name)
    manifest.append({'filename': dest_name, 'true_label': label_names[label], 'source': source})

n_glaucoma = sum(1 for m in manifest if m['true_label'] == 'glaucoma')
n_normal = sum(1 for m in manifest if m['true_label'] == 'normal')

with open(SAMPLE_DIR / 'labels.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'{len(sample_rows)} sample images copied to {SAMPLE_DIR} (glaucoma={n_glaucoma}, normal={n_normal}):')
for m in manifest:
    print(' ', m)

ZIP_PATH = '/content/glaucoma_test_samples.zip'
shutil.make_archive('/content/glaucoma_test_samples', 'zip', SAMPLE_DIR)

files.download(ZIP_PATH)   # tu dong tai ve may
print(f'
Downloaded: {ZIP_PATH}')

## SAU KHI TAI VE MAY LOCAL

1. Copy `model_best.pth` vua tai ve vao:
     D:\Fundas_System\models\glaucoma\model_best.pth
   (tao thu muc `models/glaucoma/` neu chua co)
2. Copy doan JSON in ra o Cell 8 (da co val_accuracy/val_precision/
   val_recall/val_f1 that) vao 1 file moi:
     D:\Fundas_System\config\glaucoma\config.json
3. Giai nen `glaucoma_test_samples.zip` vao 1 thu muc cuc bo (vd
   `test_images_glaucoma/`) -- dung 10 anh + `labels.json` nay de kiem thu
   thu cong sau khi viet xong `inference/glaucoma_model.py` +
   `inference/glaucoma_engine.py` (theo dung mau da lam voi multidisease,
   xem plans/plan_multidisease.txt muc 3 va plans/prep.txt muc 5).